# Programming 2C using Python

In [1]:
# Paraemter values for the 1D keyboad are stored in global variables; you can change them
pr_hit = 0.6
pr_miss = 0.4
deg_kb = 2

pr_repeat = 0.2
pr_moveOn = 0.8
deg_sp = 2

# 1. Key functions implemented so far

## getPrTableForPossibleInitialStates(lengthOfWord)

In [2]:
# C++: void getPrTableForPossibleInitialStates(lengthOfWord):
# ==> 
# Python: getPrTableForPossibleInitialStates(prTable, lengthOfWord):
#           return the information in the prTable directly
def getPrTableForPossibleInitialStates(lengthOfWord):
    missDistance = range( 1, lengthOfWord+1 )
    exponentialDegrade = [ (1/deg_sp)**i    for i in missDistance]
    scalingConstant = 1 / sum(exponentialDegrade)
    return [scalingConstant*degrade for degrade in exponentialDegrade]
    

# Test the function to get the probabilities of the possible first states
#      for a word of 3 characters (such as "his" in our handout)
pr_repeat = 0.2
pr_moveOn = 0.8
deg_sp = 2
getPrTableForPossibleInitialStates(3)


[0.5714285714285714, 0.2857142857142857, 0.14285714285714285]

## getPrTableForPossibleInitialStatesGivenTheWord(Word)

In [3]:
# A variant that accomplishes the same thing given a word (as a string)
def getPrTableForPossibleInitialStatesGivenTheWord(Word):
    missDistance = range( 1, len(Word)+1 )
    exponentialDegrade = [ (1/deg_sp)**i    for i in missDistance]
    scalingConstant = 1 / sum(exponentialDegrade)
    return [scalingConstant*degrade for degrade in exponentialDegrade]

## getPrTableForPossibleNextStates(lengthOfWord_Plus1, currentState)

In [4]:
# C++ 
# void getPrTableForPossibleNextStates(double transitionPrTable[], 
#                                      int sizeOfTable, int currentState)
# ==>
# Python 
# getPrTableForPossibleNextStates(lengthOfWord_Plus1, currentState)
# return the transitionPrTable

def getPrTableForPossibleNextStates(lengthOfWord_Plus1, currentState):
    statesAsIndices = range( lengthOfWord_Plus1 )
    distances = [state - currentState for state in statesAsIndices ]
    exponentialDegrade = [ (1/deg_sp)**i if i>0 else 0 for i in distances]
    scalingConstant = pr_moveOn/sum(exponentialDegrade)
    probabilitiesOfPossibleFirstStates = (
        [ scalingConstant*degrade for degrade in exponentialDegrade] )
    probabilitiesOfPossibleFirstStates[currentState] = pr_repeat
    return probabilitiesOfPossibleFirstStates

## getPrTableForPossibleNextStatesGivenWord(word, currentState)

In [5]:
# A convenient variant
# Python 
# getPrTableForPossibleNextStates(word, int currentState)
# return the transitionPrTable

def getPrTableForPossibleNextStatesGivenWord(word, currentState):
    lengthOfWord_Plus1 =  len(word) +1
    statesAsIndices = range( lengthOfWord_Plus1 )
    distances = [state - currentState for state in statesAsIndices ]
    exponentialDegrade = [ (1/deg_sp)**i if i>0 else 0 for i in distances]
    scalingConstant = pr_moveOn/sum(exponentialDegrade)
    probabilitiesOfPossibleFirstStates = (
        [ scalingConstant*degrade for degrade in exponentialDegrade] )
    probabilitiesOfPossibleFirstStates[currentState] = pr_repeat
    return probabilitiesOfPossibleFirstStates

## prCharGiveCharState(x, y)

In [6]:
#Probability of touching x when trying to type y
def prCharGiveCharState(x, y):
    if x==y:
        return pr_hit
    
    diffASCII = range(1,26)
    missdist = [min(n, 26-n) for n in diffASCII ]
    exponentialDegrade = [(1/deg_kb)**i for i in missdist]
    constant_x= pr_miss/sum(exponentialDegrade)
    
    distASCII_x_y = abs( ord(x) - ord(y) ) 
    distKB_x_y = min(distASCII_x_y, 26-distASCII_x_y )
    return constant_x* (1/deg_kb)** distKB_x_y 

## take1SampleFrom1PrSpace

In [7]:
# C++: int take1SampleFrom1PrSpace(double prTable[], int sizeOfTable)
# ==>
# Python: take1SampleFrom1PrSpace(prTable)
#  the size of the table can be implicitly determined 
def take1SampleFrom1PrSpace(prTable):
    probabilityThresholds = np.add.accumulate(prTable)
    sample = np.random.random()
    choice = (sample > probabilityThresholds).sum()
    # print("Sample=", sample, ",\t choice=", choice)
    return choice

## getKeyboardProbabilityTable
### note: this is a simple variant of prCharGiveCharState(x, y)

In [8]:
#C++: void getKeyboardProbabilityTable(char charToType, double prTable[])
#==>
#Python: getKeyboardProbabilityTable(charToType) 
#        to return the probabilities of getting a, b, ..., y, z
#        as a numpy array
#Note: This is simply a simple variant of prCharGiveCharState(x, y):

def getKeyboardProbabilityTable(charToType):
    #First determine the scaling constant for the exponential degrading
    diffASCII = range(1,26)
    missdist = [min(n, 26-n) for n in diffASCII ]
    exponentialDegrade = [(1/deg_kb)**i for i in missdist]
    scalingConstant = pr_miss/sum(exponentialDegrade)    
    
    #Set up an empty probability table 
    prTable = np.empty(26)
    y = charToType
    
    # for each x in a to z,
    # set up a loop to determine the probability of touching x 
    #     when trying to type y (i.e.charToType)
    # store the results in the probability table accordingly
    for i, x in enumerate("abcdefghijklmnopqrstuvwxyz"):
        if x==y:
            prTable[i] = pr_hit
        else:
            distASCII_x_y = abs( ord(x) - ord(y) ) 
            distKB_x_y = min(distASCII_x_y, 26-distASCII_x_y )
            prTable[i] = scalingConstant * (1/deg_kb)** distKB_x_y 
    
    return prTable

## typeOneChar

In [9]:
# C++: char typeOneChar(char charToType)
# Python: typeOneChar(charToType) 
#   use  take1SampleFrom1PrSpace and
#        getKeyboardProbabilityTable to
#   simulate typing charToType and return the resulting character pressed 

def typeOneChar(charToType):
    keys = "abcdefghijklmnopqrstuvwxyz" 
    prTable = getKeyboardProbabilityTable(charToType)
    indexOfKeyPressed = take1SampleFrom1PrSpace( prTable )
    return keys[ indexOfKeyPressed ]

##  typeOneWord

In [10]:
# C++: void typeOneWord( char word[], char output[], 
#                        bool traceON = false, int maxOutput=100)
# Python: typeOneWord( word, trace=False )
#      simulate the typing of the given word (a string) and
#      return the resulted string

def typeOneWord(word, trace=False):
    #Special States
    I_stateIndex = -1 
    F_stateIndex = len(word)
    
    # Step 0: Simulation of leaving the starting state I to enter some (first) state 
    #        to enter a regular state as the current state: throw a dice
    charOutputsObservedSofar = "_"
    stateTrajectorySofar = "I"
    prTable = getPrTableForPossibleInitialStatesGivenTheWord(word)

    currentState_index = take1SampleFrom1PrSpace(prTable)
    currentState_char = word[ currentState_index  ]

    if trace:
        print( "First state reached after leaving I: (index, char)=", 
               (currentState_index, currentState_char) 
             )
    
    while( currentState_index != F_stateIndex): # not the Final state F yet.
        # Step 1: Simulation of typing a character given the current state: throw a dice
        charTyped = typeOneChar(currentState_char)
        charOutputsObservedSofar += charTyped
        stateTrajectorySofar += currentState_char

        if trace:
            print( "Current state: (index, char)=", (currentState_index, currentState_char) )
            print( charTyped, " is pressed when trying to type ", currentState_char)
            print( "char outputs so far:\t", charOutputsObservedSofar )
            print( "state trajectory so far:", stateTrajectorySofar )
            print()

        # Step 2: Simulation of leaving the current state to enter one of the possble next states
        #       : throw a dice
        prTable = getPrTableForPossibleNextStatesGivenWord(word, currentState_index)
        nextState_index = take1SampleFrom1PrSpace(prTable)
        nextState_char = word[ nextState_index  ] if (nextState_index<F_stateIndex) else "F"
        if trace:
            print( "next state to enter: ", (nextState_index, 
                                             nextState_char) )

        currentState_index = nextState_index
        currentState_char = nextState_char 

    if trace:
        print("Finish typing the word ", word)
        print( "char outputs:\t\t", charOutputsObservedSofar+"_" )
        print( "state trajectory:\t", stateTrajectorySofar+"F" )
        
    return charOutputsObservedSofar[1:]

In [51]:
# Read vocabulary file
vocab = "Jumps.txt"
vFile = open(vocab)
vLines = vFile.readlines()
vFile.close()
vWords = [line.strip() for line in vLines]

vWordsUnique = []
for word in vWords:
    if word.lower() not in vWordsUnique:
        vWordsUnique.append(word.lower())

lenDistr = dict()
for l in range(1, 20):
    lenDistr[l] = 0

charDistr = dict()
for c in 'abcdefghijklmnopqrstuvwxyz':
    charDistr[c] = 0  

# Get word length and character distributions of the vocabulary
numWords = 0
for word in vWordsUnique:
    lenDistr[len(word)] += 1
    numWords += 1
    for c in word:
        charDistr[c] += 1

# 3. Implement typeOneArticle

In [47]:
import random

def typeOneArticle(corruptedMessageFile, sourceArticle, trace = False):
    outFile = open(corruptedMessageFile,"w")
    
    for i in range(49):
        wordToType = vWordsUnique[random.randint(0, numWords - 1)]
        corruptedWord = typeOneWord(wordToType)
        outFile.writelines(corruptedWord+"\n")
    outFile.close()

# Program 3A_Complete ==> 
## 1. Define a function prOf1CharSeriesWhenTyping1Word_F to generalize the process (of the forward algorithm) demonstrated in the case above for any given wordToType and any given observedString

In [13]:
# Using the forward algorithm algorithm to determine the probability: 
# The function should calculate and return
#     the probability of getting the string d 
#     when the user (modelled by the parameter values of pr_hit, pr_repeat, degenerate_kb, ...)
#     want to type the word in string w
# When the trace is True, the function will report the trace of computation done.

def prOf1CharSeriesWhenTyping1Word_F(observedString, wordToType, trace = False):
    
    #The probability distribution after leaving I
    vector_pi_list = getPrTableForPossibleInitialStatesGivenTheWord(wordToType)
    
    #The transition probability matrix A
    lenthOfWord = len(wordToType)
    matrix_A_List = [ getPrTableForPossibleNextStatesGivenWord(wordToType, currentState) 
                      for currentState in range(lenthOfWord)]
    
    #The observation probability matrix B
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    matrix_B_List = [ [prCharGiveCharState(char, state_Char) for char in alphabet] 
                      for state_Char in wordToType]
    
    ##############################################
    #Cast them into numpy arrays
    ##############################################
    vector_pi = np.array( vector_pi_list )
    matrix_A = np.array( matrix_A_List )
    matrix_B = np.array( matrix_B_List )
    
    if trace == True:
        print("vector_pi", vector_pi.shape, ":\n", vector_pi)
        print("matrix_A", matrix_A.shape, ":\n", matrix_A)
        print("matrix_B", matrix_B.shape, ":\n", matrix_B)
    
    ##############################################
    #For the first column (corresponding to the first character observed)
    ##############################################
    observationIndex = 0
    charObserved = observedString[observationIndex]
    if trace == True:
        print( "\n\nobservationIndex, charObserved:", observationIndex, ",", charObserved)
    indexOfObservedCharInAlphabet = ord(charObserved) - ord('a')
    
    #transitionProbabilties record the 1st-stage results of a column regarding 
    #    the probabilities of ending in each of the states at this point
    transitionProbabilties = vector_pi
    if trace == True:
        print("Probabilties of ending at the states at this point:\n", transitionProbabilties)
    
    #columnProbabilities record the 2nd-stage results of a column regarding 
    #    the probabilities of ending in each of the states at this point and also
    #                         seeing the specific character at this point
    observationProbabilities = matrix_B[:, indexOfObservedCharInAlphabet]
    if trace == True:
        print("probabilities of observing ", charObserved, " at specific states alone:\n", 
              observationProbabilities)

    columnProbabilities = transitionProbabilties * observationProbabilities
    if trace == True:
        print("Probabilties of observing up to", charObserved, 
              " and ending at the states at this point:\n", columnProbabilities)
    
    
    ##############################################
    # For the remaining columns one at a time
    ##############################################
    lenthOfObservedString = len(observedString)
    for observationIndex in np.arange(1, lenthOfObservedString):
        charObserved = observedString[observationIndex]
        if trace == True:
            print( "\nobservationIndex, charObserved:", observationIndex, ",", charObserved)
        #every element of columnProbabilities * every element in row i of matrix_A
        transitionProbabilties = (columnProbabilities[:, np.newaxis] * matrix_A).sum(axis = 0)
        transitionProbabilties = transitionProbabilties[:-1]  # Drop the probability to F
        if trace == True:
            print("Probabilties of ending at the states at this point:\n", transitionProbabilties)

        indexOfObservedCharInAlphabet = ord(charObserved) - ord('a')
        observationProbabilities = matrix_B[:, indexOfObservedCharInAlphabet]
        if trace == True:
            print("probabilities of observing ", charObserved, " at specific states alone:\n", 
                  observationProbabilities)

        columnProbabilities = (transitionProbabilties) * observationProbabilities
        if trace == True:
            print("Probabilties of observing up to", charObserved, 
                  " and ending at the states at this point:\n", columnProbabilities)
    
    ##############################################
    # Determine the sum of probabilities of transitioning to the Final state F from each state
    ##############################################
    if trace == True:
        print("\nprobabilities of transitioning to F from states at this point:\n", 
              matrix_A[:, len( wordToType)] )
    pr = (columnProbabilities * matrix_A[:, len( wordToType)]).sum()
    if trace == True:
        print("\nSum of the probabilitie above:", pr);
    
    return pr 


## 2. Define a function prOf1CharSeriesWhenTyping1Word_B to implement a brute-force version for any given wordToType and any given observedString

In [14]:
# A function to return the next state trajectory as a 1 dimensional numpy array
def getNextTrajectory(currentTrajectory, sizeOfStateSpace, trace = False):
    if np.any(currentTrajectory < sizeOfStateSpace-1) == False: 
         return np.zeros( currentTrajectory.shape[0] )
    
    incrementPoint = currentTrajectory.shape[0] -1
    while currentTrajectory[incrementPoint] == sizeOfStateSpace-1:
        incrementPoint -= 1
    if trace: print("incrementPoint", incrementPoint )
    nextTrajectory  = currentTrajectory.copy()
    nextTrajectory[ incrementPoint  ] += 1
    nextTrajectory[ incrementPoint+1:  ] = 0
    return nextTrajectory

In [15]:
# A more efficient variant of the function above:
#   transformToNextTrajectory simply updates the contents of current trajectory
#   to the next trajectory
def transformToNextTrajectory_x(currentTrajectory, sizeOfStateSpace, trace = False):
    if np.any(currentTrajectory < sizeOfStateSpace-1) == False: 
         return np.zeros( currentTrajectory.shape[0] )
    
    incrementPoint = currentTrajectory.shape[0] -1
    while currentTrajectory[incrementPoint] == sizeOfStateSpace-1:
        incrementPoint -= 1
    if trace: print("incrementPoint", incrementPoint )
    currentTrajectory[ incrementPoint  ] += 1
    currentTrajectory[ incrementPoint+1:  ] = 0

In [16]:
# An even more efficient variant of the function above:
#   only produce trajectories with non-decreasing state indices inside
def transformToNextTrajectory(currentTrajectory, sizeOfStateSpace, trace = False):
    if np.any(currentTrajectory < sizeOfStateSpace-1) == False: 
         return np.zeros( currentTrajectory.shape[0] )
    
    incrementPoint = currentTrajectory.shape[0] -1
    while currentTrajectory[incrementPoint] == sizeOfStateSpace-1:
        incrementPoint -= 1
    if trace: print("incrementPoint", incrementPoint )
    currentTrajectory[ incrementPoint  ] += 1
    currentTrajectory[ incrementPoint+1:  ] =  currentTrajectory[ incrementPoint  ]

In [17]:
# Using the brute-force algorithm to determine the probability: 
# The function should calculate and return
#     the probability of getting the string d 
#     when the user (modelled by the parameter values of pr_hit, pr_repeat, degenerate_kb, ...)
#     want to type the word in string w
# When the trace is True, the function will report the trace of computation done.


def prOf1CharSeriesWhenTyping1Word_B(observedString, wordToType, trace = False):
    #The probability distribution after leaving I
    vector_pi_list = getPrTableForPossibleInitialStatesGivenTheWord(wordToType)
    
    #The transition probability matrix A
    lenthOfWord = len(wordToType)
    matrix_A_List = [ getPrTableForPossibleNextStatesGivenWord(wordToType, currentState) 
                      for currentState in range(lenthOfWord)]
    
    #The observation probability matrix B
    alphabet = "abcdefghijklmnopqrstuvwxyz"
    matrix_B_List = [ [prCharGiveCharState(char, state_Char) for char in alphabet] 
                      for state_Char in wordToType]
    
    ##############################################
    #Cast them into numpy arrays
    ##############################################
    vector_pi = np.array( vector_pi_list )
    matrix_A = np.array( matrix_A_List )
    matrix_B = np.array( matrix_B_List )
    
    if trace == True:
        print("vector_pi", vector_pi.shape, ":\n", vector_pi)
        print("matrix_A", matrix_A.shape, ":\n", matrix_A)
        print("matrix_B", matrix_B.shape, ":\n", matrix_B)
    
    #The trajectory should have the same length of observedString.
    #Let's start from [0, ..., 0]
    sizeOfStateSpace = len( wordToType)
    lengthOfTrajectory = len(observedString)
    trajectory = np.zeros( lengthOfTrajectory, dtype="int32" ) 
    pr = 0
    
    allTrajectoriesExamined = False 
    while( allTrajectoriesExamined == False):
        currentStateIndex = trajectory[ 0 ]
        if trace == True:
            print("type(currentStateIndex )", type(currentStateIndex ) )
            print("currentStateIndex=", currentStateIndex )
            print("trajectory[ 0 ]=", trajectory[ 0 ] )
        prTrajectory = vector_pi[ currentStateIndex  ]

        #Check each observation 
        for observationIndex in np.arange(0, len(observedString) ): 
            currentStateIndex = trajectory[ observationIndex ]     
                
            #Multiply the observation probability
            charObserved = observedString[ observationIndex ]
            observedCharIndex =  ord(charObserved) -  ord('a')
            prTrajectory  *=  matrix_B[ currentStateIndex, observedCharIndex]

            #Check whether it is the end of observation
            if observationIndex == len(observedString)-1 : 
                nextStateIndex =  len( wordToType)      #the special final state F as the end
            else:
                nextStateIndex = trajectory[ observationIndex + 1]  # a regular next state

            #Multiply the transition probability to the next state
            prTrajectory  *=  matrix_A[ currentStateIndex, nextStateIndex]

        #Add the probability of this trajectory and observation to the total probability pr
        pr += prTrajectory
        
        transformToNextTrajectory(trajectory, sizeOfStateSpace)
        if np.all(trajectory == (sizeOfStateSpace-1) ):
            allTrajectoriesExamined = True
       
    return pr

In [18]:
import math
import numpy as np

def logPrOfGettingDocument1WhenTypingDocument2(corruptedFile):
    typedF = open(corruptedFile, "r")  # Open the typed file
    typedLines = typedF.readlines() # Read the lines
    typedF.close() # Close the file
    typedWords = [line.strip() for line in typedLines] # Cut any characters around the string line to get the words
    
    # Get a list of the probabilities of seeing each typed word given the vocabulary set
    logPrBaseE = 0
    logPrBase10 = 0
    
    # Get the probability of seeing each typed word given the vocabulary set, adding its log to the final sum(s)
    for i in range(9):
        sumPr = 0
        for j in range(numWords):
            sumPr += prOf1CharSeriesWhenTyping1Word_F(typedWords[i], vWordsUnique[j])
        logPrBaseE += math.log(sumPr)
        logPrBase10 += math.log10(sumPr)
    
    # Return the results--changed from Programming 3B so that only one value is returned. This
    #  makes the implementation of learnBestParameterValuesGivenDocument1WhenTypingDocument2 easier.
    return logPrBaseE

In [19]:
def learnBestParameterValuesGivenDocument1WhenTypingDocument2(typedDoc, sourceDoc):
    
    # Make sure that the parameter values are global (otherwise, 
    # logPrOfGettingDocument1WhenTypingDocument2 will always
    #  return the same probability)
    global pr_hit, pr_miss, pr_repeat, pr_moveOn, deg_sp, deg_kb
    
    hit = [0.3, 0.6, 0.9]
    repeat = [0.1, 0.4, 0.7]
    sp = [2, 3, 4]
    kb = [1.5, 2, 3]
    currID = 0

    # Loop through all parameter combinations
    for k in kb:
        deg_kb = k
        
        for s in sp:
            deg_sp = s
            
            for r in repeat:
                pr_repeat = r
                pr_moveOn = (1 - pr_repeat)
                
                for h in hit:
                    pr_hit = h
                    pr_miss = (1 - pr_hit)
                    
                    # Set Author 0 as most likely author on first iteration
                    if (currID == 0):
                        maxPr = logPrOfGettingDocument1WhenTypingDocument2(typedDoc)
                        bestID = currID
                        bestDegKb = deg_kb
                        bestDegSp = deg_sp
                        bestPrRepeat = pr_repeat
                        bestPrHit = pr_hit
                    else:
                        currPr = logPrOfGettingDocument1WhenTypingDocument2(typedDoc)
                        
                        # Update most likely author
                        if currPr > maxPr:
                            maxPr = currPr
                            bestID = currID
                            bestDegKb = deg_kb
                            bestDegSp = deg_sp
                            bestPrRepeat = pr_repeat
                            bestPrHit = pr_hit
                    
                    currID += 1
                    
    # Return the ID and parameters of the predicted author
    return [maxPr, bestID, bestDegKb, bestDegSp, bestPrRepeat, bestPrHit]

In [20]:
def sort(arr):
    for i in range(80):
        for j in range(80 - i):
            if (arr[j][0] < arr[j + 1][0]):
                temp = arr[j]
                arr[j] = arr[j + 1]
                arr[j + 1] = temp

In [34]:
# Print the size of the vocabulary
print("Number of words in",vocab,":",numWords)

Number of words in jumps.txt : 1


In [35]:
# Print vocabulary words
print("Vocabulary:")
for word in vWordsUnique:
    print(word,",")

Vocabulary:
jumps ,


In [36]:
# Print word length distribution
print("")
print("Word length distribution:")
for length, freq in lenDistr.items():
    print(length,":",freq)


Word length distribution:
1 : 0
2 : 0
3 : 0
4 : 0
5 : 1
6 : 0
7 : 0
8 : 0
9 : 0
10 : 0
11 : 0
12 : 0
13 : 0
14 : 0
15 : 0
16 : 0
17 : 0
18 : 0
19 : 0


In [37]:
# Print character distribution
print("")
print("Character distribution:")
for c, freq in charDistr.items():
    print(c,":",freq)


Character distribution:
a : 0
b : 0
c : 0
d : 0
e : 0
f : 0
g : 0
h : 0
i : 0
j : 1
k : 0
l : 0
m : 1
n : 0
o : 0
p : 1
q : 0
r : 0
s : 1
t : 0
u : 1
v : 0
w : 0
x : 0
y : 0
z : 0


In [52]:
kb = [1.5, 2, 3]
sp = [2, 3, 4]
repeat = [0.1, 0.4, 0.7]
hit = [0.3, 0.6, 0.9]

trueID = -1
numInTop1 = 0

# Loop through all parameter combinations
for k in kb:
    for s in sp:
        for r in repeat:
            for h in hit:
                deg_kb = k
                deg_sp = s
                
                pr_repeat = r
                pr_moveOn = 1 - pr_repeat
                
                pr_hit = h
                pr_miss = 1 - pr_hit
                
                # True author types 10 random words from the vocabulary
                trueID += 1
                typeOneArticle("X.txt", vocab)
                
                # Get the typed words
                inFile = open("X.txt", "r")
                lines = inFile.readlines()
                inFile.close()
                words = [line.strip() for line in lines]
                
                print("True author:", trueID)
                print("True parameters:", (deg_kb, deg_sp, pr_repeat, pr_hit))
                print("Words typed:", words)
                
                # Find the most likely parameter values that produced the observed strings
                prs = learnBestParameterValuesGivenDocument1WhenTypingDocument2("X.txt", vocab)
                
                bestID = prs[1]
                bestDegKb = prs[2]
                bestDegSp = prs[3]
                bestPrRepeat = prs[4]
                bestPrHit = prs[5]
                
                # Reset global parameters changed by learnBestParameters...()
                deg_kb = k
                deg_sp = s
                
                pr_repeat = r
                pr_moveOn = 1 - pr_repeat
                
                pr_hit = h
                pr_miss = 1 - pr_hit
                
                # Get Manhattan distance between the true and predicted authors' parameters
                diffKb = abs(bestDegKb - deg_kb)
                diffSp = abs(bestDegSp - deg_sp)
                diffPrRepeat = abs(bestPrRepeat - pr_repeat)
                diffPrHit = abs(bestPrHit - pr_hit)
                
                manhattanDist = diffKb + diffSp + diffPrRepeat + diffPrHit
                
                print("Predicted author:", bestID)
                print("Most likely parameters:", (bestDegKb, bestDegSp, bestPrRepeat, bestPrHit))
                print("Manhattan distance between true and most likely author:", manhattanDist)
                print("")
                
                # If the predicted author and the true author are the same, add 1 to the number of correct predictions
                if bestID == trueID:
                    numInTop1 += 1

print("% in top 1:", numInTop1/81 * 100)

True author: 0
True parameters: (1.5, 2, 0.1, 0.3)
Words typed: ['jgos', 'jj', 's', 'qvrq', 'l', 'wjr', 'jmq', 'o', 'iomx', 'jpq', 'ks', 'jmu', 'tps', 'sov', 'q', 'j', 'lmpq', 'kvmp', 'mo', 'jo', 'jt', 'yvq', 'jtq', 'pp', 'fvmm', 's', 'qis', 'mk', 'kpr', 'tnpq', 'fs', 'n', 'nmn', 'np', 'r', 'spry', 'ulms', 'want', 'ulp', 'qr', 'iuz', 'ivq', 'hgmq', 'ms', 'jutq', 'ov', 'suip', 's', 'janots']
Predicted author: 0
Most likely parameters: (1.5, 2, 0.1, 0.3)
Manhattan distance between true and most likely author: 0.0

True author: 1
True parameters: (1.5, 2, 0.1, 0.6)
Words typed: ['uumms', 'uumgp', 'jpw', 'us', 'cm', 'jms', 'utmp', 'r', 'urst', 'jus', 'jp', 'mptu', 'nmmpq', 'njm', 'tsumps', 'junnp', 'vp', 'jmpt', 'm', 'jxpo', 'jts', 'juvs', 'wmw', 'mps', 'ss', 'ipq', 'jjum', 'jxqu', 'pk', 'uujm', 'jbmp', 'jus', 'unpkr', 'vms', 'jisp', 'uup', 'm', 'mps', 'ju', 'jjjgos', 'u', 'pn', 'g', 'uppr', 'kou', 'nvs', 'umq', 'uos', 'us']
Predicted author: 1
Most likely parameters: (1.5, 2, 0.1, 0.6)
Ma

Predicted author: 3
Most likely parameters: (1.5, 2, 0.4, 0.3)
Manhattan distance between true and most likely author: 1.0

True author: 13
True parameters: (1.5, 3, 0.4, 0.6)
Words typed: ['hwxx', 'juzqappswr', 'uqo', 'htxukrossxrso', 'ohpsu', 'umns', 'umppn', 'jjhimp', 'mp', 'kuupra', 'uqo', 'djqfnmqnmp', 'hjums', 'mppt', 'utmpsumk', 'jppw', 'mp', 'jumpmv', 'juuss', 'jkkums', 'kjguvqmjmkpps', 'jir', 'rsnv', 'jjsuuus', 'ju', 'luossu', 'jmmopr', 'ituustuuur', 'lmnsrcrs', 'mmm', 'jjurprs', 'jtqt', 'jp', 'vmtrs', 'juwtors', 'jmmujss', 'jmqs', 'tnnpsyp', 'mlmcqpps', 'usrcp', 'jus', 'juympltx', 'muvrplmmmps', 'jjummw', 'jjumpps', 'fummmkossm', 'v', 'iuvors', 'ismhppp']
Predicted author: 13
Most likely parameters: (1.5, 3, 0.4, 0.6)
Manhattan distance between true and most likely author: 0.0

True author: 14
True parameters: (1.5, 3, 0.4, 0.9)
Words typed: ['umsssss', 'uupsss', 'jup', 'jjjjummgss', 'jjjuuppppsss', 'jurmpprs', 'rumnpps', 'jusssv', 'umppppss', 'np', 'juuo', 'junpss', 'uusps',

Predicted author: 24
Most likely parameters: (1.5, 4, 0.7, 0.3)
Manhattan distance between true and most likely author: 0.0

True author: 25
True parameters: (1.5, 4, 0.7, 0.6)
Words typed: ['uuomppyqq', 'ejmuuknsusssst', 'jjuuuuuxuvmjppo', 'jjjuummmjmmmomss', 'jjjhmrksommmsppppurs', 'sm', 'jruuumwppsst', 'jsjr', 'nuvuuuuxssqr', 'juuuppmm', 'jjjsuumpns', 'jjrjjjirjumhms', 'jpuuykuua', 'ijjjwuxuutp', 'djwnmmqokpqppppnppqo', 'jjjuuvuywmmstsss', 'jjjjjjjulnppmoppprorrrppppqpopr', 'jummlmm', 'krupvssps', 'jjjummmmnp', 'ijjrjjjjrjuppotsso', 'ijjtuuukopvuqsssr', 'jfhglflkjjunsroppnpmpss', 'ulsqssss', 'eijjjjiuvuuuuuuupnmrp', 'jjjmmtpsmmmvrqp', 'juuumtpstxxsu', 'mjhkjsppqqlpnsvstsss', 'ipjiijnkjmjjgjjjjxujsms', 'nlfjjkjjuuqssrvrrs', 'jjki', 'uojsipprsr', 'pnppvsssr', 'jjemmmppnppppppps', 'jjjdnqjhjjsdmqrmprqqzsrsv', 'kjkjjjfqsuuutnpmmlsqs', 'jjklhutm', 'jjiuuzuutrykpppsssrsrp', 'lkjjppqppppjssqvs', 'joklolspppps', 'xsymmqpruss', 'uuukmpis', 'nkhpuuimkss', 'jtlnrpp', 'jjjjjdjuuuuvquutvrmmpptss

Predicted author: 54
Most likely parameters: (3, 2, 0.1, 0.3)
Manhattan distance between true and most likely author: 2.0

True author: 37
True parameters: (2, 3, 0.1, 0.6)
Words typed: ['jn', 'kps', 'ump', 'jjjums', 'ipt', 'jwp', 'imps', 'ps', 'jujs', 'jums', 'jups', 'ps', 'jtkpr', 'jvppss', 'ukms', 'jqmw', 'uq', 'lmr', 'wmmp', 'jkumus', 'jkou', 'jmqq', 'jtmqpr', 'ss', 'hwllt', 'jws', 'humos', 'ums', 'fjumqs', 'jujp', 'ljps', 'iumps', 'ks', 'lums', 'jpqt', 'juk', 'jujps', 'unp', 'ums', 'jvp', 'uupr', 'jkq', 'mrss', 'ljuump', 'msmpx', 'juupt', 'jus', 'hopr', 'jumqs']
Predicted author: 55
Most likely parameters: (3, 2, 0.1, 0.6)
Manhattan distance between true and most likely author: 2.0

True author: 38
True parameters: (2, 3, 0.1, 0.9)
Words typed: ['jups', 'juuqs', 'jmp', 'um', 'jmms', 'mps', 'jmn', 'jum', 'umps', 'jtps', 'jjmps', 'ups', 'jus', 'mps', 'j', 'vps', 'umsps', 'ps', 'pmpu', 'jmps', 'mps', 'um', 'jymmps', 'js', 'kum', 'jumpvs', 'jmr', 'um', 'lpt', 'umr', 'umpt', 'iqs', 'ju

Predicted author: 66
Most likely parameters: (3, 3, 0.4, 0.3)
Manhattan distance between true and most likely author: 2.0

True author: 49
True parameters: (2, 4, 0.4, 0.6)
Words typed: ['u', 'ppps', 'lumnpps', 'pmsss', 'pumlmms', 'hwmp', 'yuuulkppssss', 'nwt', 'iuumpsssssr', 'jmjnpst', 'suppp', 'juposs', 'gunrv', 'hijmnps', 'fkijijmpqs', 'jwuumps', 'ilhitmmls', 'jukkts', 'jgummn', 'hjnuvq', 'ulnssps', 'mjps', 'jusrs', 'jjguxls', 'jinuss', 'jjvplrs', 'jjompus', 'jijlp', 'uumqts', 'k', 'umimmppy', 'jjkmmimmkqp', 'hums', 'kjlhmjjsss', 'ms', 'jjupqp', 'iwukr', 'rss', 'rnms', 'lkjjkvlpsr', 'jamkmpq', 'jvlmss', 'jsps', 'jumls', 'jtummp', 'kjjjtumnss', 'iiquopru', 'tvtuzuups', 'ujmllpppruq']
Predicted author: 13
Most likely parameters: (1.5, 3, 0.4, 0.6)
Manhattan distance between true and most likely author: 1.5

True author: 50
True parameters: (2, 4, 0.4, 0.9)
Words typed: ['jummmps', 'jumpsssss', 'kmqs', 'kuuuump', 'jjjjukpps', 'jjvmps', 'jummp', 'juuuns', 'jjjuusss', 'jup', 'jjummpss', 

Predicted author: 60
Most likely parameters: (3, 2, 0.7, 0.3)
Manhattan distance between true and most likely author: 0.0

True author: 61
True parameters: (3, 2, 0.7, 0.6)
Words typed: ['unpr', 'jjkuuvssst', 'jumppopqppqpnr', 'nsu', 'ijuutvtusptss', 't', 'ljjumnmsrstsst', 'jkurposss', 'mlls', 'jlnjmjmmnpssqss', 'ipqt', 'jrptppqoopps', 't', 'uvpsr', 'kjijtpmops', 'fjjjmmmpppp', 'jrtsts', 'ijku', 'usrlmrs', 'tvuuposspss', 'jtr', 'jvpqppqp', 'smnklrsst', 'ruvuopppspsus', 'jmrssrv', 'jutuuututuusssrrrttrssr', 'unommmmnopmoppq', 'mmlrrs', 'nn', 'jjjkivts', 'mmoqpssssstrs', 'ss', 'urr', 'ij', 'jjjijuuus', 'mjijkmsu', 'vmoppppnopsrs', 'jjljmuvtuvuuussustusss', 'ulssuuss', 'juuopp', 'kvmppo', 'kkjjuuuus', 'kkkikuuuuquummmnspopnppppq', 'kulmms', 'srstssutswsss', 'mmnmos', 'jjjnlmmmlpstr', 'opopsvssqrs', 'ljuuuuswumnnnnlllm']
Predicted author: 61
Most likely parameters: (3, 2, 0.7, 0.6)
Manhattan distance between true and most likely author: 0.0

True author: 62
True parameters: (3, 2, 0.7, 0.9

Predicted author: 62
Most likely parameters: (3, 2, 0.7, 0.9)
Manhattan distance between true and most likely author: 1.0

True author: 72
True parameters: (3, 4, 0.1, 0.3)
Words typed: ['mvvmqq', 'hus', 'jtlqs', 'ukos', 'umtr', 'ups', 'gnmqu', 'uums', 'juuoqs', 'lwnpr', 'nt', 'kwmqq', 'kumnr', 'nt', 'kvmqr', 'ivmnr', 'ivnru', 'jsnor', 'mvmow', 'junpr', 'jiwss', 'ujrs', 'ivnmt', 'unrpr', 'kvoss', 'hkloqp', 'jops', 'ivloq', 'kr', 'jzlpu', 'kvk', 'tmqt', 'vnu', 'kuqs', 'voqr', 'kqr', 'inn', 'kkslmsq', 'ivpqu', 'jvlpo', 'kjvr', 'vs', 'jor', 'iulot', 'imp', 'iuopqps', 'nr', 'jlorss', 'lkpq']
Predicted author: 73
Most likely parameters: (3, 4, 0.1, 0.6)
Manhattan distance between true and most likely author: 0.3

True author: 73
True parameters: (3, 4, 0.1, 0.6)
Words typed: ['jumps', 'jumlsu', 'juuump', 'juonp', 'jmpls', 'ns', 'hmms', 'jtnmlmo', 'jmpq', 'umpt', 'dtmpt', 'mr', 'jxpu', 'jivlnpr', 'mqs', 'jup', 'tmps', 'jum', 'hurps', 'unmr', 'jujs', 'mup', 'tpss', 'ums', 'jjumps', 'jtmt', 'k